In [103]:
!pip install yfinance pandas-gbq -q

In [104]:
from google.colab import auth
auth.authenticate_user()

## Importação dos Dados

In [ ]:
# Bibliotecas
import yfinance as yf
import pandas as pd

# Ações a serem baixadas
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN","NVDA",]

# Loop para baixar todas as ações no mesmo espaço de tempo
df_all = []
for ticker in tickers:
    df = yf.download(ticker, start="2010-01-01")
    df.columns = df.columns.get_level_values(0)
    df["ticker"] = ticker
    df.reset_index(inplace=True)
    df_all.append(df)

# Compacta as ações em um único dataframe
df_final = pd.concat(df_all, ignore_index=True)
df = df_final.copy()

/tmp/ipykernel_9540/1775150596.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2010-01-01")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_9540/1775150596.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2010-01-01")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_9540/1775150596.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2010-01-01")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_9540/1775150596.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2010-01-01")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_9540/1775150596.py:9: FutureWarning: YF.download() has change

In [ ]:
    print(f'\n {df.shape}')


 (20600, 7)


## Limpeza dos Dados

In [107]:
df_final.head()

Price,Date,Close,High,Low,Open,Volume,ticker
0,2010-01-04,6.406479,6.421148,6.357685,6.389117,493729600,AAPL
1,2010-01-05,6.417558,6.453780,6.383730,6.424144,601904800,AAPL
2,2010-01-06,6.315476,6.443001,6.308890,6.417556,552160000,AAPL
3,2010-01-07,6.303801,6.346309,6.258000,6.338825,477131200,AAPL
4,2010-01-08,6.345712,6.346311,6.258301,6.295420,447610800,AAPL


In [108]:
df.isnull().sum()

,0
Price,
Date,0
Close,0
High,0
Low,0
Open,0
Volume,0
ticker,0


In [109]:
df = df_final.sort_values("Date")

In [110]:
df[df.duplicated(subset=["Date"], keep=False)]

Price,Date,Close,High,Low,Open,Volume,ticker
0,2010-01-04,6.406479,6.421148,6.357685,6.389117,493729600,AAPL
16480,2010-01-04,0.423784,0.426763,0.415074,0.424242,800204000,NVDA
4120,2010-01-04,23.077387,23.189232,22.808958,22.831328,38409100,MSFT
12360,2010-01-04,6.695000,6.830500,6.657000,6.812500,151998000,AMZN
8240,2010-01-04,15.555865,15.624369,15.493568,15.560829,78169752,GOOGL
...,...,...,...,...,...,...,...
16479,2026-05-20,264.719910,265.579987,259.532990,260.049988,21931807,AMZN
8239,2026-05-20,419.880005,421.160004,411.300110,413.985992,17748105,MSFT
4119,2026-05-20,302.190094,302.799988,298.089996,298.179993,23148975,AAPL
12359,2026-05-20,387.220001,393.859894,382.899994,387.695007,18917407,GOOGL


In [111]:
df = df.groupby("Date").agg({
    "Open": "first",
    "High": "max",
    "Low": "min",
    "Close": "last",
    "Volume": "sum"
}).reset_index()

In [112]:
df = df.sort_values("Date")

In [113]:
df.dropna()

Price,Date,Open,High,Low,Close,Volume
0,2010-01-04,6.389117,23.189232,0.415074,15.555865,1562510452
1,2010-01-05,0.422179,23.189229,0.422179,23.084839,1677408212
2,2010-01-06,15.533774,23.174307,0.425617,6.315476,1562075252
3,2010-01-07,15.125238,22.890968,0.421033,0.424242,1552402328
4,2010-01-08,14.693373,23.025186,0.418283,22.861147,1362369228
...,...,...,...,...,...,...
4115,2026-05-14,229.850006,411.839996,229.300003,298.209991,295660400
4116,2026-05-15,262.500000,428.170013,224.240005,421.920013,347691600
4117,2026-05-18,300.239990,425.119995,218.369995,264.859985,273856000
4118,2026-05-19,429.899994,432.700012,217.910004,298.970001,295071600


In [114]:
df.head()

Price,Date,Open,High,Low,Close,Volume
0,2010-01-04,6.389117,23.189232,0.415074,15.555865,1562510452
1,2010-01-05,0.422179,23.189229,0.422179,23.084839,1677408212
2,2010-01-06,15.533774,23.174307,0.425617,6.315476,1562075252
3,2010-01-07,15.125238,22.890968,0.421033,0.424242,1552402328
4,2010-01-08,14.693373,23.025186,0.418283,22.861147,1362369228


In [115]:
df_final.to_gbq(
    destination_table="stocks_data.raw_prices",
    project_id="stock-predictor-496217",
    if_exists="replace"
)

/tmp/ipykernel_9540/2311732967.py:1: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_final.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 4957.81it/s]


## Feature engineering

In [116]:
from google.cloud import bigquery
client = bigquery.Client(project="stock-predictor-496217")
df = client.query("SELECT * FROM stocks_data.raw_prices ORDER BY ticker, Date").to_dataframe()

In [117]:
# def classify_return(x):

#     if x < -0.02:
#         return 0   # queda forte

#     elif x > 0.02:
#         return 2   # subida forte

#     else:
#         return 1   # lateral

def add_features(df):

    # RSI
    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()
    rs = avg_gain / avg_loss

    df["RSI"] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = df["Close"].ewm(span=12).mean()
    ema26 = df["Close"].ewm(span=26).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_signal"] = df["MACD"].ewm(span=9).mean()

    # Bollinger Bands
    sma20 = df["Close"].rolling(20).mean()
    std20 = df["Close"].rolling(20).std()
    df["BB_upper"] = sma20 + 2 * std20
    df["BB_lower"] = sma20 - 2 * std20
    df["BB_position"] = (df["Close"] - df["BB_lower"]) / (df["BB_upper"] - df["BB_lower"])

    # Para colocar % - Diminui Ruído
    df["return_1"] = df["Close"].pct_change(1)
    df["return_5"] = df["Close"].pct_change(5)
    df["return_10"] = df["Close"].pct_change(10)

    # média dos preços, mas dando MAIS peso aos candles recentes.
    df["EMA_9"] = df["Close"].ewm(span=9).mean()
    df["EMA_21"] = df["Close"].ewm(span=21).mean()
    df["EMA_50"] = df["Close"].ewm(span=50).mean()
    df["trend"] = (df["EMA_9"] > df["EMA_21"]).astype(int)

    df["volatility"] = df["Close"].pct_change().rolling(20).std()

    df["bull_market"] = (
        df["EMA_21"] > df["EMA_50"]
    ).astype(int)

    # Target (1 = subiu, 0 = caiu)
    future_return = (df["Close"].shift(-5) - df["Close"]) / df["Close"]
    df["target"] = (future_return > 0.03).astype(int)

    # df["target"] = future_return.apply(classify_return)

    return df


In [118]:
df = df.groupby("ticker", group_keys=False).apply(add_features)

/tmp/ipykernel_9540/1754811764.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("ticker", group_keys=False).apply(add_features)


In [119]:
df = df.dropna()

In [120]:
(f'Valores nulos:')
df.isnull().sum()

,0
Date,0
Close,0
High,0
Low,0
Open,0
Volume,0
ticker,0
RSI,0
MACD,0
MACD_signal,0


In [121]:
df.to_gbq(
    destination_table="stocks_data.features",
    project_id="stock-predictor-496217",
    if_exists="replace"
)

/tmp/ipykernel_9540/2071502382.py:1: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 9020.01it/s]
